In [300]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time
from functools import partial

In [284]:
import random
import math

In [285]:
random.seed(42)

In [286]:
tests = ['tsp_51_1', 'tsp_100_3', 'tsp_200_2', 'tsp_574_1', 'tsp_1889_1', 'tsp_33810_1']
thresholds = [(482, 430), (23433, 20800), (35985, 30000), (40000, 37600), (378069, 323000), (78478868, 67700000)]

In [287]:
def load_points(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n = int(lines[0].strip())
        points = list()
        for i in range(n):
            x, y = list(map(float, lines[1 + i].split()))
            points.append((x, y))

        return n, points

In [288]:
def check_tsp(n, points, order):
    used = [0] * n
    for i in order:
        if i >= n or i < 0:
            raise Exception("Not correct ordering")
            
        used[i] += 1
        if used[i] >= 2:
            raise Exception("Not correct ordering")
            
    result = 0
    for i in range(n):
        nxt = i + 1
        if nxt == n:
            nxt = 0

        pt1 = points[order[i]]
        pt2 = points[order[nxt]]
        
        result += math.dist(pt1, pt2)
        
    return result

In [289]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [290]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, points = load_points(test)
        start = time.time()
        
        if not use_file:
            order = method(n, points)
        else:
            order = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_tsp(n, points, order)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Напишем сначала наиболее простое решение, которое просто находит жадный порядок: всегда берет самое близкую вершину.

In [291]:
!g++ -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [292]:
def greedy_tsp(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        order = list(map(int, lines[0].split()))
        return order

In [293]:
test_method(greedy_tsp, "greedy", True)

Checking greedy
Execution time: 0.3760 seconds
Target function tsp_51_1: 506.363165362846
Passed tsp_51_1: 0
Execution time: 0.0041 seconds
Target function tsp_100_3: 25138.785452772318
Passed tsp_100_3: 0
Execution time: 0.0041 seconds
Target function tsp_200_2: 36226.22143814145
Passed tsp_200_2: 0
Execution time: 0.0083 seconds
Target function tsp_574_1: 47054.9474467063
Passed tsp_574_1: 0
Execution time: 0.0446 seconds
Target function tsp_1889_1: 391470.44549188914
Passed tsp_1889_1: 0
Execution time: 9.5213 seconds
Target function tsp_33810_1: 78478867.03022148
Passed tsp_33810_1: 1
Score: 3


Прошел только один тест, поэтому давайте это решение подтюним. А именно будем пробовать сделать reverse подотрезков в каком - то порядке, чтобы улучшить ответ и так делать, пока можем. Все это делается на жадном решении, как некотором приближении.

In [294]:
!g++ -std=c++2a cpp_methods/greedy_local_opt.cpp -o tmp/greedy_local_opt

In [295]:
def greedy_local_opt_tsp(test_file): 
    os.system(f"./tmp/greedy_local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        order = list(map(int, lines[0].split()))
        return order

In [296]:
test_method(greedy_local_opt_tsp, "greedy_local_opt", True)

Checking greedy_local_opt
Execution time: 0.1728 seconds
Target function tsp_51_1: 440.15503849930053
Passed tsp_51_1: 1
Execution time: 0.0065 seconds
Target function tsp_100_3: 21746.189521377997
Passed tsp_100_3: 1
Execution time: 0.0305 seconds
Target function tsp_200_2: 31499.538298693467
Passed tsp_200_2: 1
Execution time: 0.4321 seconds
Target function tsp_574_1: 39538.653372590314
Passed tsp_574_1: 1
Execution time: 13.6636 seconds
Target function tsp_1889_1: 341010.65746338386
Passed tsp_1889_1: 1
Execution time: 70.5169 seconds
Target function tsp_33810_1: 78012119.5601017
Passed tsp_33810_1: 1
Score: 18


Прошли все простые пороги, 2-opt действительно неплохо работает в задаче TSP

Теперь рассмотрим естественное улучшение этого подхода. Будем брать текущее лучшее найденное решение и применять к нему несколько случайных модификаций, а именно K случайных реверсов подотрезков. После этого будем снова запускать локальное улучшение 2-opt, чтобы привести полученное решение к локальному оптимуму. (LNS)

In [298]:
!g++ -std=c++2a cpp_methods/random_local_opt.cpp -o tmp/random_local_opt

In [299]:
def random_local_opt_tsp(test_file, K=20): 
    os.system(f"./tmp/random_local_opt data/{test_file} {K}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        order = list(map(int, lines[0].split()))
        return order

In [301]:
test_method(partial(random_local_opt_tsp, K=3), "random_local_opt", True)

Checking random_local_opt
Execution time: 101.3492 seconds
Target function tsp_51_1: 428.87175639203383
Passed tsp_51_1: 2
Execution time: 101.0073 seconds
Target function tsp_100_3: 20750.76250368752
Passed tsp_100_3: 2
Execution time: 101.0391 seconds
Target function tsp_200_2: 29455.02098403309
Passed tsp_200_2: 2
Execution time: 101.6399 seconds
Target function tsp_574_1: 37810.32932558621
Passed tsp_574_1: 1
Execution time: 116.8469 seconds
Target function tsp_1889_1: 332925.7407652126
Passed tsp_1889_1: 1
Execution time: 197.0289 seconds
Target function tsp_33810_1: 78007083.4239766
Passed tsp_33810_1: 1
Score: 24


In [302]:
test_method(partial(random_local_opt_tsp, K=5), "random_local_opt", True)

Checking random_local_opt
Execution time: 101.0046 seconds
Target function tsp_51_1: 428.87175639203383
Passed tsp_51_1: 2
Execution time: 101.0082 seconds
Target function tsp_100_3: 20750.762503687525
Passed tsp_100_3: 2
Execution time: 101.0669 seconds
Target function tsp_200_2: 29561.599984307006
Passed tsp_200_2: 2
Execution time: 101.8865 seconds
Target function tsp_574_1: 38680.54658853018
Passed tsp_574_1: 1
Execution time: 115.0517 seconds
Target function tsp_1889_1: 336545.42551508243
Passed tsp_1889_1: 1
Execution time: 196.1590 seconds
Target function tsp_33810_1: 77991395.14821588
Passed tsp_33810_1: 1
Score: 24


In [303]:
test_method(partial(random_local_opt_tsp, K=12), "random_local_opt", True)

Checking random_local_opt
Execution time: 101.0050 seconds
Target function tsp_51_1: 428.87175639203383
Passed tsp_51_1: 2
Execution time: 101.0129 seconds
Target function tsp_100_3: 20750.76250368754
Passed tsp_100_3: 2
Execution time: 101.0483 seconds
Target function tsp_200_2: 29754.582107680526
Passed tsp_200_2: 2
Execution time: 101.7723 seconds
Target function tsp_574_1: 38735.31090982305
Passed tsp_574_1: 1
Execution time: 115.3738 seconds
Target function tsp_1889_1: 339702.02149614727
Passed tsp_1889_1: 1
Execution time: 195.9393 seconds
Target function tsp_33810_1: 77991395.14821588
Passed tsp_33810_1: 1
Score: 24


Получился значительный прирост результатов, а теперь попробуем воспользоваться другим случайным изменением структуры. А именно методом double-bridge. 
Разбиваем на 5 путей:
1. $[0, a)$
2. $[a, b)$
3. $[b, c)$
4. $[c, d)$
5. $[d, n)$

И склеиваем заменой:
1. $[0, a)$
2. $[c, d)$
3. $[b, c)$
4. $[a, b)$
5. $[d, n)$

То есть, переставляем два отрезка местами.

In [276]:
!g++ -O2 -std=c++2a cpp_methods/tuned_local_opt.cpp -o tmp/tuned_local_opt

In [277]:
def tuned_local_opt_tsp(test_file): 
    os.system(f"./tmp/tuned_local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        order = list(map(int, lines[0].split()))
        return order

In [279]:
test_method(tuned_local_opt_tsp, "tuned_local_opt", True)

Checking tuned_local_opt
Execution time: 601.0081 seconds
Target function tsp_51_1: 428.98164717220675
Passed tsp_51_1: 2
Execution time: 601.0467 seconds
Target function tsp_100_3: 20750.76250368752
Passed tsp_100_3: 2
Execution time: 601.0042 seconds
Target function tsp_200_2: 29440.412220624265
Passed tsp_200_2: 2
Execution time: 601.0225 seconds
Target function tsp_574_1: 37051.09660346339
Passed tsp_574_1: 2
Execution time: 601.8835 seconds
Target function tsp_1889_1: 323580.5845027866
Passed tsp_1889_1: 1
Execution time: 685.6155 seconds
Target function tsp_33810_1: 76864999.73843436
Passed tsp_33810_1: 1
Score: 26


Прошел еще один тест, но дальше нужно пробовать либо другие стратегии изменения решения или по - другому его достраивать (не через 2-opt).

Попробуем реализовать алгоритм Кернигана-Лина аналогично тому, что было на семинаре. А именно делать ejection chains.

In [280]:
!g++ -O2 -std=c++2a cpp_methods/kernighan_lin.cpp -o tmp/kernighan_lin

In [281]:
def kernighan_lin_tsp(test_file): 
    os.system(f"./tmp/kernighan_lin data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        order = list(map(int, lines[0].split()))
        return order

In [282]:
test_method(kernighan_lin_tsp, "kernighan_lin", True)

Checking kernighan_lin
Execution time: 601.3626 seconds
Target function tsp_51_1: 428.87175639203383
Passed tsp_51_1: 2
Execution time: 600.9984 seconds
Target function tsp_100_3: 20750.76250368752
Passed tsp_100_3: 2
Execution time: 601.0337 seconds
Target function tsp_200_2: 29440.973826897352
Passed tsp_200_2: 2
Execution time: 601.0242 seconds
Target function tsp_574_1: 37258.50321060081
Passed tsp_574_1: 2
Execution time: 601.3881 seconds
Target function tsp_1889_1: 320917.27065505163
Passed tsp_1889_1: 2
Execution time: 622.1434 seconds
Target function tsp_33810_1: 72876249.11260752
Passed tsp_33810_1: 1
Score: 28


Результаты улучшились, но даже за 10 минут, не получилось пробить последний порог.

Итог:

1. Простое жадное решение показывает достаточно слабые метрики.

2. Его можно существенно улучшить с помощью локального поиска: если последовательно применять реверсы отрезков до тех пор, пока они улучшают ответ, то удаётся пройти все простые пороги.

3. Добавление стохастики заметно повышает качество решения. Если делать K случайных реверсов, тем самым немного «портить» текущее решение, а затем снова применять к нему локальную оптимизацию, то начинают проходиться уже более сложные пороги. Фактически получается некоторый LNS.

4. При этом возникает свобода выбора того, как именно портить текущее решение. В качестве альтернативы я попробовал преобразование double-bridge, которое также дало улучшение.

5. Наконец, в качестве наиболее сильного подхода был реализован алгоритм Кернигана — Лина с ejection chains и большим числом рестартов. При запуске примерно на 10 минут на каждый тест этот метод позволил пройти все пороги, кроме последнего.